In [ ]:
import io
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 120)


def load_dataset():
    """
    Load the movie dataset.

    The dataset is embedded directly in this script as a CSV string so the
    program runs standalone with no external file dependency. It is
    structured the same way as the popular TMDB 5000 Movies dataset
    (title, genres, keywords, overview, cast, director), so you can swap in
    a real downloaded CSV later by replacing this function with:

        movies = pd.read_csv("movies.csv")

    and the rest of the pipeline will work unchanged.
    """
    csv_data = """movie_id,title,genres,keywords,overview,cast,director
1,Inception,Action Adventure Sci-Fi Thriller,dream heist subconscious corporate espionage layered reality,A skilled thief who steals secrets from people's subconscious during dreams is given a chance to have his criminal history erased if he can implant an idea into a target's mind.,Leonardo DiCaprio Joseph Gordon-Levitt Ellen Page Tom Hardy,Christopher Nolan
2,Interstellar,Adventure Drama Sci-Fi,space travel wormhole time dilation black hole family survival,A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival as Earth becomes uninhabitable.,Matthew McConaughey Anne Hathaway Jessica Chastain,Christopher Nolan
3,The Dark Knight,Action Crime Drama Thriller,vigilante joker gotham chaos moral choice,Batman raises the stakes in his war on crime battling the menace known as the Joker who plunges Gotham into anarchy.,Christian Bale Heath Ledger Aaron Eckhart,Christopher Nolan
4,The Prestige,Drama Mystery Sci-Fi Thriller,magician rivalry obsession illusion revenge,Two stage magicians engage in a battle to create the ultimate illusion while sacrificing everything they have to outwit each other.,Christian Bale Hugh Jackman Scarlett Johansson,Christopher Nolan
5,Memento,Mystery Thriller,amnesia revenge memory loss tattoo investigation,A man with short-term memory loss attempts to track down his wife's murderer using notes and tattoos.,Guy Pearce Carrie-Anne Moss Joe Pantoliano,Christopher Nolan
6,The Matrix,Action Sci-Fi,virtual reality simulation hacker rebellion machines,A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers.,Keanu Reeves Laurence Fishburne Carrie-Anne Moss,Lana Wachowski
7,The Matrix Reloaded,Action Sci-Fi,virtual reality rebellion machines prophecy war,Neo and the rebel leaders estimate that they have 72 hours until the machines discover the underground city of Zion.,Keanu Reeves Laurence Fishburne Carrie-Anne Moss,Lana Wachowski
8,John Wick,Action Crime Thriller,assassin revenge dog underworld retired hitman,An ex-hitman comes out of retirement to track down the gangsters that took everything from him.,Keanu Reeves Michael Nyqvist Alfie Allen,Chad Stahelski
9,John Wick: Chapter 2,Action Crime Thriller,assassin revenge underworld blood debt mercenary,John Wick is forced to back out of retirement by a former associate looking to seize control of an international assassins' guild.,Keanu Reeves Common Laurence Fishburne,Chad Stahelski
10,Avengers: Endgame,Action Adventure Sci-Fi,superhero time travel sacrifice universe rescue,The Avengers assemble once more to undo Thanos' actions and restore order to the universe.,Robert Downey Jr. Chris Evans Scarlett Johansson,Anthony Russo
11,Avengers: Infinity War,Action Adventure Sci-Fi,superhero villain universe sacrifice infinity stones,The Avengers and their allies must be willing to sacrifice all in an attempt to defeat the powerful Thanos.,Robert Downey Jr. Chris Hemsworth Mark Ruffalo,Anthony Russo
12,Iron Man,Action Adventure Sci-Fi,superhero billionaire armor weapons redemption,After being held captive in an Afghan cave billionaire engineer Tony Stark creates a unique weaponized suit of armor to fight evil.,,Jon Favreau
13,Iron Man 2,Action Adventure Sci-Fi,superhero villain armor government weapons,Tony Stark must contend with deadly issues involving the government his own friends as well as new enemies due to the public knowledge of his status as Iron Man.,Robert Downey Jr. Mickey Rourke Gwyneth Paltrow,Jon Favreau
14,Spider-Man: Homecoming,Action Adventure Sci-Fi,superhero teenager school villain mentor,Peter Parker balances being a high school student and a superhero as he uncovers a new threat to New York City.,Tom Holland Michael Keaton Robert Downey Jr.,Jon Watts
15,The Lion King,Animation Adventure Drama,lion kingdom betrayal coming of age savannah,Lion prince Simba flees his kingdom after the murder of his father only to learn the true meaning of responsibility and bravery.,Matthew Broderick Jeremy Irons James Earl Jones,Roger Allers
16,Finding Nemo,Animation Adventure Family,ocean father son search fish friendship,After his son is captured in the Great Barrier Reef and taken to Sydney a timid clownfish sets out on a journey to bring him home.,Albert Brooks Ellen DeGeneres Alexander Gould,Andrew Stanton
17,Finding Dory,Animation Adventure Family,ocean memory loss family search fish,Dory a forgetful blue tang fish is on a journey to reunite with her parents learning more about herself along the way.,Ellen DeGeneres Albert Brooks Ed O'Neill,Andrew Stanton
18,Toy Story,Animation Adventure Comedy Family,toys friendship jealousy rescue childhood,A cowboy doll is profoundly threatened and jealous when a new spaceman action figure supplants him as top toy in a boy's room.,Tom Hanks Tim Allen Don Rickles,John Lasseter
19,Toy Story 3,Animation Adventure Comedy Family,toys daycare escape friendship growing up,The toys are mistakenly delivered to a daycare center and must plan an escape before being given away by the now grown up Andy.,Tom Hanks Tim Allen Joan Cusack,Lee Unkrich
20,Up,Animation Adventure Comedy Drama Family,balloons adventure grief house wilderness,Seventy-eight year old Carl Fredricksen travels to Paradise Falls in his house equipped with balloons inadvertently taking a young stowaway.,Edward Asner Jordan Nagai Christopher Plummer,Pete Docter
21,WALL-E,Animation Adventure Family Sci-Fi,robot earth waste love space,In the distant future a small waste-collecting robot inadvertently embarks on a space journey that will ultimately decide the fate of mankind.,Ben Burtt Elissa Knight Jeff Garlin,Andrew Stanton
22,The Shawshank Redemption,Drama,prison friendship hope wrongful conviction redemption,Two imprisoned men bond over a number of years finding solace and eventual redemption through acts of common decency.,Tim Robbins Morgan Freeman Bob Gunton,Frank Darabont
23,The Godfather,Crime Drama,mafia family power crime succession patriarch,The aging patriarch of an organized crime dynasty transfers control of his clandestine empire to his reluctant son.,Marlon Brando Al Pacino James Caan,Francis Ford Coppola
24,The Godfather Part II,Crime Drama,mafia family power crime flashback empire,The early life and career of Vito Corleone in 1920s New York is portrayed while his son Michael expands the family business.,Al Pacino Robert De Niro Robert Duvall,Francis Ford Coppola
25,Goodfellas,Biography Crime Drama,mafia gangster loyalty betrayal organized crime,The story of Henry Hill and his life in the mob covering his relationship with his wife and his partners in crime.,Robert De Niro Ray Liotta Joe Pesci,Martin Scorsese
26,Pulp Fiction,Crime Drama,gangster nonlinear narrative violence redemption diner,The lives of two mob hitmen a boxer a gangster's wife and a pair of diner bandits intertwine in four tales of violence and redemption.,John Travolta Uma Thurman Samuel L. Jackson,Quentin Tarantino
27,Kill Bill: Vol. 1,Action Crime Thriller,revenge assassin sword bride martial arts,After awakening from a four-year coma a former assassin wreaks vengeance on the team of assassins who betrayed her.,Uma Thurman David Carradine Daryl Hannah,Quentin Tarantino
28,Reservoir Dogs,Crime Drama Thriller,heist betrayal undercover gangster diner,A group of criminals who don't know each other's real identities or backgrounds come together to commit a diamond heist.,Harvey Keitel Tim Roth Michael Madsen,Quentin Tarantino
29,Fight Club,Drama,insomnia secret society anarchy identity underground,An insomniac office worker and a devil-may-care soap maker form an underground fight club that evolves into something much more.,Brad Pitt Edward Norton Meat Loaf,David Fincher
30,Se7en,Crime Drama Mystery Thriller,serial killer detective sin investigation grim city,Two detectives one retiring and one new on the job hunt a serial killer who uses the seven deadly sins as his motives.,Morgan Freeman Brad Pitt Kevin Spacey,David Fincher
31,Gone Girl,Drama Mystery Thriller,missing wife marriage investigation media frenzy,With his wife's disappearance having become the focus of an intense media circus a man sees the spotlight turned on him when it's suspected that he may not be innocent.,Ben Affleck Rosamund Pike Neil Patrick Harris,David Fincher
32,The Social Network,Biography Drama,internet startup lawsuit friendship betrayal college,The story of the founders of Facebook and the lawsuits that followed the social network's success.,Jesse Eisenberg Andrew Garfield Justin Timberlake,David Fincher
33,Titanic,Drama Romance,shipwreck romance class divide tragedy ocean liner,A seventeen-year-old aristocrat falls in love with a kind but poor artist aboard the luxurious ill-fated R.M.S. Titanic.,Leonardo DiCaprio Kate Winslet Billy Zane,James Cameron
34,Avatar,Action Adventure Fantasy Sci-Fi,alien planet colonization nature war indigenous tribe,A paraplegic Marine dispatched to the moon Pandora becomes torn between following his orders and protecting the world he feels is his home.,Sam Worthington Zoe Saldana Sigourney Weaver,James Cameron
35,Terminator 2: Judgment Day,Action Sci-Fi,cyborg time travel robot apocalypse protection,A cyborg identical to the one who failed to kill Sarah Connor must now protect her teenage son from a more advanced cyborg.,Arnold Schwarzenegger Linda Hamilton Edward Furlong,James Cameron
36,Forrest Gump,Drama Romance,life journey vietnam war love simplicity history,The presidency of Lyndon B. Johnson through to the 1980s unfolds from the perspective of an Alabama man with an extraordinarily low IQ.,Tom Hanks Robin Wright Gary Sinise,Robert Zemeckis
37,Back to the Future,Adventure Comedy Sci-Fi,time travel delorean past family teenager,A teenager is accidentally sent thirty years into the past in a time-traveling car invented by his eccentric scientist friend.,Michael J. Fox Christopher Lloyd Lea Thompson,Robert Zemeckis
38,Cast Away,Drama,plane crash survival island isolation solitude,A FedEx executive must transform himself physically and emotionally to survive a crash landing on a deserted island.,Tom Hanks Helen Hunt,Robert Zemeckis
39,Gladiator,Action Adventure Drama,rome revenge empire arena betrayal warrior,A former Roman general sets out to exact vengeance against the corrupt emperor who murdered his family and sent him into slavery.,Russell Crowe Joaquin Phoenix Connie Nielsen,Ridley Scott
40,Alien,Horror Sci-Fi,spaceship monster survival crew isolation,The crew of a commercial spacecraft encounter a deadly lifeform after investigating an unknown transmission.,Sigourney Weaver Tom Skerritt John Hurt,Ridley Scott
41,Blade Runner,Sci-Fi Thriller,,A blade runner must pursue and terminate four replicants who stole a ship in space and have returned to Earth seeking their creator.,Harrison Ford Rutger Hauer Sean Young,Ridley Scott
42,The Martian,Adventure Drama Sci-Fi,mars survival astronaut isolation rescue science,An astronaut becomes stranded on Mars and must use his ingenuity to survive while his crew devises a daring rescue mission.,Matt Damon Jessica Chastain Kristen Wiig,Ridley Scott
43,Saving Private Ryan,Drama War,world war normandy rescue mission soldiers brotherhood,Following the Normandy landings a group of soldiers go behind enemy lines to retrieve a paratrooper whose brothers have been killed in action.,Tom Hanks Matt Damon Edward Burns,Steven Spielberg
44,Jurassic Park,Adventure Sci-Fi Thriller,dinosaurs theme park genetic engineering survival island,A pragmatic paleontologist visiting an almost complete theme park is tasked with protecting visitors after a power failure causes the park's cloned dinosaurs to run loose.,Sam Neill Laura Dern Jeff Goldblum,Steven Spielberg
45,Jaws,Adventure Horror Thriller,shark beach town summer fear ocean,When a killer shark unleashes chaos on a beach community it's up to a local sheriff a marine biologist and an old seafarer to hunt the beast down.,Roy Scheider Robert Shaw Richard Dreyfuss,Steven Spielberg
46,Schindler's List,Biography Drama History,world war holocaust factory rescue moral awakening,In German-occupied Poland during World War II industrialist Oskar Schindler gradually becomes concerned for his Jewish workforce.,Liam Neeson Ralph Fiennes Ben Kingsley,Steven Spielberg
47,The Departed,Crime Drama Thriller,undercover mob police boston betrayal,An undercover cop and a mole in the police attempt to identify each other while infiltrating an Irish gang in South Boston.,Leonardo DiCaprio Matt Damon Jack Nicholson,Martin Scorsese
48,Shutter Island,Mystery Thriller,asylum investigation conspiracy island detective,A U.S. Marshal investigates the disappearance of a patient from a hospital for the criminally insane on an isolated island.,Leonardo DiCaprio Mark Ruffalo Ben Kingsley,Martin Scorsese
49,The Wolf of Wall Street,Biography Comedy Crime Drama,stock market fraud excess wealth corruption,Based on the true story of Jordan Belfort his career on Wall Street and how it led to crimes in connection to corruption and fraud.,Leonardo DiCaprio Jonah Hill Margot Robbie,Martin Scorsese
50,Joker,Crime Drama Thriller,clown mental illness gotham chaos comedian,In Gotham City mentally troubled comedian Arthur Fleck is disregarded and mistreated by society leading him on a downward spiral.,Joaquin Phoenix Robert De Niro Zazie Beetz,Todd Phillips
51,Spirited Away,Animation Adventure Family Fantasy,spirit world bathhouse magic coming of age,During her family's move to the suburbs a sullen 10-year-old girl wanders into a world ruled by gods witches and spirits.,Rumi Hiiragi Miyu Irino Mari Natsuki,Hayao Miyazaki
52,My Neighbor Totoro,Animation Family Fantasy,forest spirit childhood countryside friendship,Two sisters move to the countryside with their father and discover the surrounding nature is inhabited by friendly forest spirits.,Hitoshi Takagi Noriko Hidaka,Hayao Miyazaki
53,Coco,Animation Adventure Family Fantasy,afterlife family music tradition mexico memory,Aspiring musician Miguel enters the Land of the Dead to find his great-great-grandfather a legendary singer.,Anthony Gonzalez Gael Garcia Bernal,Lee Unkrich
54,Inside Out,Animation Adventure Comedy Drama Family,emotions childhood memory growing up mind,After young Riley is uprooted from her Midwest life her emotions conflict on how best to navigate a new city house and school.,Amy Poehler Phyllis Smith Bill Hader,Pete Docter
55,Zootopia,Animation Adventure Comedy Family,animals city detective prejudice mystery,In a city of anthropomorphic animals a rookie bunny cop and a cynical con artist fox must work together to uncover a conspiracy.,Ginnifer Goodwin Jason Bateman Idris Elba,Byron Howard
56,The Incredibles,Animation Action Adventure Family ,superhero family secret identity villain teamwork,A family of undercover superheroes while trying to live the quiet suburban life are forced into action to save the world.,Craig T. Nelson Holly Hunter Samuel L. Jackson,Brad Bird
57,Shrek,Animation Adventure Comedy Family Fantasy,ogre fairy tale rescue swamp kingdom,A mean lord exiles fairytale creatures to the swamp of a grumpy ogre who must go on a quest and rescue a princess to get his land back.,Mike Myers Eddie Murphy Cameron Diaz,Andrew Adamson
58,Harry Potter and the Sorcerer's Stone,Adventure Family Fantasy,wizard school magic friendship destiny,An orphaned boy enrolls in a school of wizardry where he learns the truth about himself his family and the terrible evil that haunts the magical world.,Daniel Radcliffe Rupert Grint Emma Watson,Chris Columbus
59,Harry Potter and the Chamber of Secrets,Adventure Family Fantasy,wizard school monster mystery friendship,Harry ignores warnings not to return to Hogwarts only to find the school plagued by a series of mysterious attacks.,Daniel Radcliffe Rupert Grint Emma Watson,Chris Columbus
60,The Lord of the Rings: The Fellowship of the Ring,Adventure Fantasy,ring quest fellowship middle earth dark lord,A meek hobbit and eight companions set out on a journey to destroy a powerful ring and save Middle-earth from the Dark Lord Sauron.,Elijah Wood Ian McKellen Orlando Bloom,Peter Jackson
61,The Lord of the Rings: The Two Towers,Adventure Fantasy,ring quest war middle earth alliance,While Frodo and Sam edge closer to Mordor the divided fellowship faces a mounting threat from the dark forces of Sauron.,Elijah Wood Ian McKellen Viggo Mortensen,Peter Jackson
62,The Hobbit: An Unexpected Journey,Adventure Fantasy,dwarves quest dragon treasure journey,A reluctant hobbit sets out to the Lonely Mountain with a spirited group of dwarves to reclaim their mountain home from a dragon.,Ian McKellen Martin Freeman Richard Armitage,Peter Jackson
63,Star Wars: A New Hope,Action Adventure Fantasy Sci-Fi,space empire rebellion jedi farm boy destiny,Luke Skywalker joins forces with a Jedi Knight to rescue a princess and save the galaxy from the Empire's world-destroying battle station.,Mark Hamill Harrison Ford Carrie Fisher,George Lucas
64,Star Wars: The Empire Strikes Back,Action Adventure Fantasy Sci-Fi,space empire jedi training betrayal rebellion,After the rebels are overpowered Luke Skywalker begins Jedi training while his friends are pursued by Darth Vader.,Mark Hamill Harrison Ford Carrie Fisher,Irvin Kershner
65,Guardians of the Galaxy,Action Adventure Comedy Sci-Fi,space heist team superhero outlaws humor,A group of intergalactic criminals must pull together to stop a fanatical warrior with plans to purge the universe.,Chris Pratt Zoe Saldana Dave Bautista,James Gunn
66,Doctor Strange,Action Adventure Fantasy,magic surgeon multiverse sorcerer mystical,A brilliant surgeon has his world shattered when an accident destroys his hands leading him to a path of magic and mysticism.,Benedict Cumberbatch Chiwetel Ejiofor Rachel McAdams,Scott Derrickson
67,Black Panther,Action Adventure Sci-Fi,superhero kingdom africa succession technology,T'Challa returns home to the isolated technologically advanced African nation of Wakanda to take his place as king but a powerful enemy reappears.,Chadwick Boseman Michael B. Jordan Lupita Nyong'o,Ryan Coogler
68,Wonder Woman,Action Adventure Fantasy Sci-Fi,superhero amazon warrior world war origin,When a pilot crashes and tells of conflict in the outside world Diana an Amazon warrior in training leaves home to fight a war.,Gal Gadot Chris Pine Robin Wright,Patty Jenkins
69,Logan,Action Drama Sci-Fi,mutant aging road trip protection dystopia,In a future where mutants are nearly extinct an aging Wolverine must protect a young mutant girl from dark forces.,Hugh Jackman Patrick Stewart Dafne Keen,James Mangold
70,Mad Max: Fury Road,Action Adventure Sci-Fi,wasteland chase warlord rebellion survival,A woman rebels against a tyrannical warlord in postapocalyptic Australia in search for her homeland with the aid of a group of female prisoners and a drifter named Max.,Tom Hardy Charlize Theron Nicholas Hoult,George Miller
71,No Country for Old Men,Crime Drama Thriller,hitman pursuit money texas violence fate,Violence and mayhem ensue after a hunter stumbles upon a drug deal gone wrong and a satchel of cash in the desert.,Tommy Lee Jones Javier Bardem Josh Brolin,Joel Coen
72,There Will Be Blood,Drama,oil tycoon ambition greed religion frontier,A story of family religion hatred and oil centered around a turn-of-the-century prospector in the early days of the business.,Daniel Day-Lewis Paul Dano,Paul Thomas Anderson
73,12 Years a Slave,Biography Drama History,slavery freedom injustice survival south,A free black man is kidnapped and sold into slavery where he must endure years of hardship before regaining his freedom.,Chiwetel Ejiofor Michael Fassbender Lupita Nyong'o,Steve McQueen
74,Moonlight,Drama,identity sexuality childhood coming of age miami,A young African-American man grapples with his identity and sexuality while experiencing the everyday struggles of childhood adolescence and adulthood.,Trevante Rhodes Andre Holland Janelle Monae,Barry Jenkins
75,The Grand Budapest Hotel,Adventure Comedy Crime Drama,hotel concierge heist europe whimsical caper,A concierge at a famous hotel between the wars and a young employee form a friendship during a struggle for an inheritance amid a caper involving a stolen painting.,Ralph Fiennes Tony Revolori Saoirse Ronan,Wes Anderson
76,Eternal Sunshine of the Spotless Mind,Drama Romance Sci-Fi,memory erasure love heartbreak relationship,A couple undergoes a procedure to have their memories of each other erased after they break up.,Jim Carrey Kate Winslet Kirsten Dunst,Michel Gondry
77,The Truman Show,Comedy Drama,reality show surveillance simulated town awakening,An insurance salesman discovers his entire life is actually a reality TV show and that everyone around him is an actor.,Jim Carrey Ed Harris Laura Linney,Peter Weir
78,Her,Drama Romance Sci-Fi,artificial intelligence loneliness love technology,A lonely writer develops an unlikely relationship with an operating system designed to meet his every need.,Joaquin Phoenix Scarlett Johansson Amy Adams,Spike Jonze
79,Ex Machina,Drama Sci-Fi Thriller,artificial intelligence robot isolation manipulation test,A young programmer is selected to evaluate the human qualities of a breathtaking humanoid AI but ulterior motives lurk beneath the test.,Domhnall Gleeson Alicia Vikander Oscar Isaac,Alex Garland
80,Arrival,Drama Mystery Sci-Fi,alien language communication linguist time,A linguist is recruited by the military to communicate with alien lifeforms after twelve mysterious spacecrafts appear around the world.,Amy Adams Jeremy Renner Forest Whitaker,Denis Villeneuve
81,Dune,Adventure Drama Sci-Fi,desert planet prophecy empire spice rebellion,A noble family becomes embroiled in a war for control over the galaxy's most valuable asset on a treacherous desert planet.,Timothee Chalamet Rebecca Ferguson Zendaya,Denis Villeneuve
82,Blade Runner 2049,Drama Sci-Fi Thriller,android dystopia detective future identity replicant,A young blade runner's discovery of a long-buried secret leads him to track down former blade runner Rick Deckard.,Ryan Gosling Harrison Ford Ana de Armas,Denis Villeneuve
83,Prisoners,Crime Drama Mystery Thriller,kidnapping investigation family suburb desperation,When two young girls go missing a desperate father takes matters into his own hands as the police pursue multiple leads.,Hugh Jackman Jake Gyllenhaal Viola Davis,Denis Villeneuve
84,Sicario,Action Crime Drama Thriller,cartel agent border violence morality,An idealistic FBI agent is enlisted by a government task force to bring down a drug cartel leader along the U.S.-Mexico border.,Emily Blunt Josh Brolin Benicio del Toro,Denis Villeneuve
85,The Prestige,Drama Mystery Sci-Fi Thriller,magician rivalry obsession illusion revenge,Two stage magicians engage in a battle to create the ultimate illusion while sacrificing everything they have to outwit each other.,Christian Bale Hugh Jackman Scarlett Johansson,Christopher Nolan
"""
    movies = pd.read_csv(io.StringIO(csv_data))
    print(f"Dataset loaded: {movies.shape[0]} rows, {movies.shape[1]} columns\n")
    print(movies.head(3))
    return movies


def explore_and_clean(movies):
    """
    Explore the dataset for missing values / duplicates, then clean it:
      - drop duplicate movie titles
      - fill missing text fields with an empty string (no row needs to be
        dropped just because one optional field is blank)
      - strip stray whitespace from text columns
    """
    print("\n--- Exploring dataset ---")
    print("Missing values per column:")
    print(movies.isna().sum())
    print("\nDuplicate titles found:", movies.duplicated(subset="title").sum())

    print("\n--- Cleaning dataset ---")
    movies = movies.drop_duplicates(subset="title").reset_index(drop=True)

    text_cols = ["genres", "keywords", "overview", "cast", "director"]
    movies[text_cols] = movies[text_cols].fillna("")
    for col in text_cols + ["title"]:
        movies[col] = movies[col].str.strip()

    assert movies["title"].is_unique, "Titles are not unique after cleaning!"
    print(f"Cleaned dataset shape: {movies.shape}")
    print("Remaining missing values:", movies.isna().sum().sum())
    return movies


def extract_features(movies):
    """
    Combine genres, keywords, director, cast, and overview into a single
    'tags' column used for similarity matching.

    Genres and keywords are repeated twice on purpose: they are short,
    precise descriptors, while the overview is full-sentence prose where
    word overlap between two unrelated movies is often coincidental.
    Repeating the precise tags gives them more influence once vectorized,
    which produces noticeably better recommendations.
    """
    def make_tags(row):
        return " ".join([
            row["genres"], row["genres"],
            row["keywords"], row["keywords"],
            row["director"],
            row["cast"],
            row["overview"],
        ])

    movies["tags"] = movies.apply(make_tags, axis=1).str.lower()
    print("Sample combined feature text (Inception):")
    print(movies.loc[movies["title"] == "Inception", "tags"].values[0])
    return movies



def vectorize_text(movies):
    """
    Convert the 'tags' text column into TF-IDF numerical vectors.
    Stop words ("the", "and", "is", ...) are removed since they carry no
    signal about what a movie is actually about.
    """
    tfidf = TfidfVectorizer(stop_words="english", max_features=5000)
    tfidf_matrix = tfidf.fit_transform(movies["tags"])
    print(f"TF-IDF matrix shape: {tfidf_matrix.shape} "
          f"({tfidf_matrix.shape[0]} movies x {tfidf_matrix.shape[1]} vocabulary terms)")
    return tfidf_matrix


def calculate_similarity(tfidf_matrix):
    """
    Compute pairwise cosine similarity between every movie's TF-IDF vector.
    Cosine similarity measures the angle between vectors, so it isn't biased
    by movies simply having longer text fields.
    """
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    print(f"Similarity matrix shape: {cosine_sim.shape}")
    assert np.allclose(cosine_sim.diagonal(), 1.0), "Each movie should be perfectly similar to itself."
    print("Diagonal check passed (every movie has similarity 1.0 with itself).")
    return cosine_sim


def build_title_index(movies):
    """Map lowercase title -> dataframe row index, for fast lookup."""
    return pd.Series(movies.index, index=movies["title"].str.lower())


def find_movie_index(title, title_to_index):
    """
    Resolve a user-provided title to a row index.
    Tries an exact (case-insensitive) match first, then falls back to a
    substring match so partial titles still work. Returns None if nothing matches.
    """
    title_clean = title.strip().lower()
    if title_clean in title_to_index:
        return title_to_index[title_clean]

    partial_matches = [t for t in title_to_index.index if title_clean in t]
    if partial_matches:
        return title_to_index[partial_matches[0]]

    return None


def recommend_movies(title, movies, cosine_sim, title_to_index, top_n=5):
    """
    Recommend the top_n movies most similar to `title`.

    Steps: resolve title -> index, look up its similarity row, sort
    descending, drop the movie itself, return the top_n as a DataFrame.
    """
    idx = find_movie_index(title, title_to_index)
    if idx is None:
        return f"'{title}' was not found in the dataset. Please check the spelling and try again."

    matched_title = movies.loc[idx, "title"]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda pair: pair[1], reverse=True)
    sim_scores = [pair for pair in sim_scores if pair[0] != idx][:top_n]

    result_indices = [pair[0] for pair in sim_scores]
    result_scores = [round(pair[1], 4) for pair in sim_scores]

    recommendations = movies.loc[result_indices, ["title", "genres", "director"]].copy()
    recommendations.insert(0, "rank", range(1, len(recommendations) + 1))
    recommendations["similarity_score"] = result_scores
    recommendations = recommendations.reset_index(drop=True)

    print(f"\nBecause you watched: {matched_title}\n")
    return recommendations


def run_tests(movies, cosine_sim, title_to_index):
    test_titles = [
        "Inception",
        "The Dark Knight",
        "Toy Story",
        "The Matrix",
        "dark knight",
        "The Avengers Strike Back 5",
    ]

    for title in test_titles:
        print("=" * 70)
        result = recommend_movies(title, movies, cosine_sim, title_to_index, top_n=5)
        print(result)



def print_observations():
    print("\n" + "=" * 70)
    print("OBSERVATIONS AND RESULTS")
    print("=" * 70)
    print("""
1. This is a CONTENT-BASED recommender: it compares what movies are about
   (genres, keywords, cast, director, overview), not what other users rated.

2. Weighting genres/keywords more heavily (counted twice in the combined
   text) produced clearly better recommendations than treating every field
   equally - e.g. Inception correctly surfaces The Matrix, and The Dark
   Knight correctly surfaces Joker.

3. Similarity scores are naturally low in absolute terms (often 0.05-0.35).
   This is expected with TF-IDF + cosine similarity on short text - what
   matters is the RELATIVE ranking, not the raw score.

4. Sequels and same-director films cluster tightly, since they share cast,
   director, and keyword fields almost exactly (e.g. Toy Story -> Toy Story 3).

5. Limitation: if a title isn't in the dataset (or is too different to
   partial-match), the system can't recommend anything for it - a known
   limitation of any catalog-based content recommender.

Possible improvements: use a larger real-world dataset (e.g. full TMDB 5000),
experiment with different feature weightings, add a popularity/rating
tie-breaker, or combine with collaborative filtering for a hybrid system.
""")



def main():
    movies = load_dataset()
    movies = explore_and_clean(movies)
    movies = extract_features(movies)
    tfidf_matrix = vectorize_text(movies)
    cosine_sim = calculate_similarity(tfidf_matrix)
    title_to_index = build_title_index(movies)

    run_tests(movies, cosine_sim, title_to_index)
    print_observations()

    print("=" * 70)
    print("Type a movie title to get recommendations (or press Enter to quit).")
    while True:
        user_input = input("\nEnter a movie title: ").strip()
        if not user_input:
            print("Goodbye!")
            break
        print(recommend_movies(user_input, movies, cosine_sim, title_to_index, top_n=5))


if __name__ == "__main__":
    main()

Dataset loaded: 85 rows, 7 columns

   movie_id            title                            genres  \
0         1        Inception  Action Adventure Sci-Fi Thriller   
1         2     Interstellar            Adventure Drama Sci-Fi   
2         3  The Dark Knight       Action Crime Drama Thriller   

                                                      keywords  \
0  dream heist subconscious corporate espionage layered rea...   
1  space travel wormhole time dilation black hole family su...   
2                    vigilante joker gotham chaos moral choice   

                                                      overview  \
0  A skilled thief who steals secrets from people's subcons...   
1  A team of explorers travel through a wormhole in space i...   
2  Batman raises the stakes in his war on crime battling th...   

                                                          cast           director  
0  Leonardo DiCaprio Joseph Gordon-Levitt Ellen Page Tom Hardy  Christopher Nolan  
1

KeyboardInterrupt: Interrupted by user